In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [167]:
df = pd.read_csv('train.txt', sep = ';' , header = None , names = ['text','emotion'])

# Apply all text cleaning steps sequentially to ensure correct processing
df['text'] = df['text'].apply(lambda x : x.lower())
df['text'] = df['text'].apply(remove_punc)
df['text'] = df['text'].apply(remove_numbers)
df['text'] = df['text'].apply(remove_emojis)
df['text'] = df['text'].apply(remove)

In [117]:
import os
os.listdir()

['.config', 'train.txt', 'sample_data']

In [118]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [119]:
df.isnull().sum()

,0
text,0
emotion,0


In [120]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [121]:
unique_emotion = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotion:
  emotion_numbers[emo]=i
  i +=1

df['emotion']=df['emotion'].map(emotion_numbers)

In [122]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [123]:
df['text']=df['text'].apply(lambda x : x.lower())

In [124]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [125]:
df['text'] = df['text'].apply(remove_punc)

In [126]:
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

df['text']=df['text'].apply(remove_numbers)

In [139]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii(): # Keep ASCII characters
      new += i
  return new

df['text']=df['text'].apply(remove_emojis)

In [128]:
import nltk

In [129]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [130]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [141]:
def remove(txt):
  words = word_tokenize(txt)
  cleaned = []
  for i in words:
    if i not in stop_words:
      cleaned.append(i)
  return' '.join(cleaned)

In [132]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [133]:
df['text'] = df['text']

In [134]:
df.loc[1]['text']

''

In [135]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)


In [145]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [146]:
bow_vectorizer = CountVectorizer()



In [149]:
X_train_processed = X_train.apply(lambda x: x if x != '' else "<EMPTY_TEXT>")
X_test_processed = X_test.apply(lambda x: x if x != '' else "<EMPTY_TEXT>")

X_train_bow = bow_vectorizer.fit_transform(X_train_processed)
X_test_bow = bow_vectorizer.transform(X_test_processed)

In [150]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [151]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [153]:
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))


0.3190625


In [154]:
pred_bow

array([5, 5, 5, ..., 5, 5, 5])

In [155]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_processed)
X_test_tfidf = tfidf_vectorizer.transform(X_test_processed)

In [156]:
nb2_moded= MultinomialNB()
nb2_moded.fit(X_train_tfidf, y_train)

MultinomialNB()

In [158]:
y_pred=nb2_moded.predict(X_test_tfidf)

In [159]:
print(accuracy_score(y_test, y_pred))

0.3190625


In [160]:
from sklearn.linear_model import LogisticRegression

In [164]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [165]:
log_pred = logistic_model.predict(X_test_tfidf)

In [166]:
print(accuracy_score(y_test, log_pred))

0.3190625
